# 3.3 LangChain Ecosystem Overview

## Playground Notebook

In this notebook, we'll explore the **LangChain ecosystem** hands-on using a **local Ollama model**.

| Topic | What You'll Learn |
|-------|-------------------|
| **What is LangChain** | The framework, its purpose, and the problem it solves |
| **The Five Pillars** | Core, LangChain, Community/Partners, LangGraph, LangSmith |
| **Core Building Blocks** | Models, Prompts, Chains, Output Parsers |
| **LCEL (Pipe Syntax)** | Composing components with the `\|` operator |
| **Memory** | Conversation history management |
| **Swappability** | How LangChain abstractions let you swap providers easily |

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [ ]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama langchain-community

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick test — make sure Ollama is running
test = llm.invoke("Say 'hello' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Test response: {test.content}")

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Connected to Ollama | Model: llama3.2:3b
   Test response: Hello


---

## 1. Why LangChain? — The Problem It Solves

LLMs by themselves are **stateless text generators**. To build real applications you need:

- Prompt management
- Chaining multiple reasoning steps
- Connecting to external data
- Conversation memory
- Tool orchestration
- Error handling

**Without LangChain** — a Document Q&A system requires 1000+ lines of custom, tightly coupled code.

**With LangChain** — the same system can be built with ~30 lines of composable, modular code.

### The Five Pillars of the LangChain Ecosystem

| Pillar | Package | Purpose |
|--------|---------|----------|
| **1. Core** | `langchain-core` | Base abstractions, Runnable protocol, LCEL, message types |
| **2. LangChain** | `langchain` | Chains, agents, memory modules — cognitive architectures |
| **3. Community/Partners** | `langchain-ollama`, etc. | 700+ integrations — LLM providers, vector DBs, tools |
| **4. LangGraph** | `langgraph` | Stateful agents with graph-based control flow |
| **5. LangSmith** | `langsmith` | Observability — debugging, testing, monitoring |

Let's explore the most important ones hands-on.

---

## 2. Pillar 1 — LangChain Core: Message Types & the Runnable Protocol

Everything in LangChain implements the **Runnable protocol** — a universal interface with:

| Method | What It Does |
|--------|--------------|
| `invoke()` | Process a single input |
| `stream()` | Get output token by token |
| `batch()` | Process multiple inputs at once |

LangChain uses structured **message types** instead of raw dicts:

```
SystemMessage   →  Sets behavior, persona, constraints
HumanMessage    →  The user's input
AIMessage       →  The model's response
```

### Experiment 2A: Using Message Types Directly

In [3]:
# LangChain message types — structured, typed, and self-documenting
messages = [
    SystemMessage(content="You are a concise technical explainer. Max 2 sentences."),
    HumanMessage(content="What is LangChain?")
]

response = llm.invoke(messages)

print(f"Type: {type(response).__name__}")
print(f"Content: {response.content}")

Type: AIMessage
Content: LangChain is an open-source Python library that provides a simple and flexible way to chain together multiple language models, such as text classification, sentiment analysis, and question answering tasks. It allows users to create custom workflows by combining different models and fine-tuning them on their specific use case data.

LangChain enables users to:

* Chain multiple model layers for complex tasks
* Fine-tune pre-trained models on their own dataset
* Integrate with other libraries like Transformers and Hugging Face

It supports a range of pre-trained models, including BERT, RoBERTa, DistilBERT, and more. LangChain's core advantage is its ability to provide a high-level abstraction for complex NLP workflows, making it easier to build and deploy custom models without requiring extensive expertise in deep learning or model architecture.


### Experiment 2B: The Runnable Protocol — `invoke()`, `stream()`, `batch()`

In [4]:
# Every LangChain component supports these three methods

# --- invoke: single input, full response ---
print("=" * 50)
print("invoke() — Single input, full response")
print("=" * 50)
result = llm.invoke("What is Python? One sentence.")
print(result.content)

invoke() — Single input, full response
Python is a high-level, interpreted programming language that is widely used for various applications such as web development, scientific computing, data analysis, artificial intelligence, and more, known for its simplicity, readability, and large community of developers.


In [5]:
# --- stream: token by token ---
print("=" * 50)
print("stream() — Tokens arrive as generated")
print("=" * 50)

for chunk in llm.stream("Name 3 programming languages. Brief."):
    print(chunk.content, end="", flush=True)
print()

stream() — Tokens arrive as generated
Here are three programming languages:

1. Python
2. JavaScript
3. C++


In [6]:
# --- batch: multiple inputs at once ---
print("=" * 50)
print("batch() — Process multiple inputs")
print("=" * 50)

questions = [
    "What is an API? One sentence.",
    "What is a vector database? One sentence.",
    "What is RAG? One sentence."
]

results = llm.batch(questions)

for q, r in zip(questions, results):
    print(f"Q: {q}")
    print(f"A: {r.content}\n")

batch() — Process multiple inputs
Q: What is an API? One sentence.
A: An API (Application Programming Interface) is a set of defined rules that allows different software systems to communicate with each other, enabling data exchange, functionality sharing, and integration between applications.

Q: What is a vector database? One sentence.
A: A vector database is a data storage system that indexes and retrieves dense vector representations of entities, such as text documents or images, using efficient algorithms like k-Nearest Neighbors (k-NN) searches to facilitate fast similarity matching.

Q: What is RAG? One sentence.
A: RAG (Relation-Aware Graph) is a type of neural network architecture that uses graph-based methods to model complex relationships between entities in text, such as entities mentioned in sentences or passages, and their relationships with each other.



---

## 3. Prompt Templates — Reusable, Parameterized Prompts

Instead of building prompt strings manually, LangChain provides **ChatPromptTemplate** — a reusable template with variables.

```python
# Manual string building (fragile, error-prone)
prompt = f"Explain {topic} to a {audience}"

# LangChain template (reusable, composable)
template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}."),
    ("human", "Explain {topic} to a {audience}.")
])
```

### Experiment 3A: Creating and Using Prompt Templates

In [7]:
# Define a reusable prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Keep answers to 2-3 sentences."),
    ("human", "Explain {topic} to a {audience}.")
])

# Format with specific values
messages = prompt.invoke({
    "role": "friendly teacher",
    "topic": "LangChain",
    "audience": "beginner programmer"
})

# See what was generated
print("Formatted messages:")
for msg in messages.messages:
    print(f"  [{type(msg).__name__}] {msg.content}")

print("\nResponse:")
response = llm.invoke(messages)
display(Markdown(response.content))

Formatted messages:
  [SystemMessage] You are a friendly teacher. Keep answers to 2-3 sentences.
  [HumanMessage] Explain LangChain to a beginner programmer.

Response:


LangChain is an open-source Python library that makes it easy to build custom AI workflows by chaining together different natural language processing (NLP) models and tools.

Think of a workflow like a recipe. Just as you need to follow a specific order of ingredients, cooking time, and temperature to make a dish, an NLP model needs to process text in a certain way to extract useful information. LangChain helps you build these workflows by providing a simple, intuitive API that lets you chain together different components.

Here's an analogy to help illustrate the concept:

Imagine you're building a chatbot that needs to do the following tasks:

1. Understand the user's input (e.g., text analysis)
2. Look up synonyms for specific words (e.g., word embeddings)
3. Translate the text into another language
4. Summarize the translated text

With LangChain, you could create a workflow like this:
```
Input -> TextAnalysis -> WordEmbeddings -> Translation -> Summarization -> Output
```
LangChain provides pre-built components for each of these tasks, which can be easily chained together using its API. This makes it easy to build complex NLP workflows without having to write custom code.

Some key benefits of LangChain include:

* **Modularity**: LangChain allows you to build workflows composed of smaller, reusable components.
* **Flexibility**: You can customize each component to fit your specific use case.
* **Scalability**: LangChain makes it easy to add new components or workflows as needed.

Overall, LangChain is a powerful tool for building custom NLP workflows, and its simplicity makes it accessible to developers of all skill levels.

In [8]:
# Reuse the SAME template with different inputs
scenarios = [
    {"role": "scientist", "topic": "vector embeddings", "audience": "high school student"},
    {"role": "chef", "topic": "APIs", "audience": "someone who only knows cooking"},
]

for scenario in scenarios:
    print(f"\n{'=' * 50}")
    print(f"Role: {scenario['role']} | Topic: {scenario['topic']}")
    print(f"{'=' * 50}")
    messages = prompt.invoke(scenario)
    response = llm.invoke(messages)
    display(Markdown(response.content))


Role: scientist | Topic: vector embeddings


Imagine you're trying to describe a friend to someone who's never met them before. You might say something like, "My friend is tall and has brown hair." But how would you represent your friend as a single unit that captures all their characteristics?

One way to do this is by creating a **vector**. A vector is like an arrow in space with different lengths and directions on each side. Each direction represents a characteristic of your friend, like height or hair color.

In the case of word embeddings, we create vectors for words (like "tall" or "brown") that capture their meanings. These vectors are like arrows in a huge space where all possible combinations of words exist.

Here's an example:

* The vector for "tall" might be [1, 0, 0] (meaning it's long and thin)
* The vector for "brown" might be [0, 1, 1] (meaning it's got a lot of color)

To combine these vectors into a new vector that represents the word "my friend is tall", we need to multiply them together. This gives us a new vector that captures both characteristics: [1, 0, 0] × [0, 1, 0] = [0, 1, 0].

The resulting vector for "tall" is combined with the vectors for other words to create a representation of your friend's entire description. This process is called **vector addition**.

Here are some key things about vector embeddings:

* **Similarity**: Two words that have similar meanings will have similar vectors.
* **Distance**: The distance between two vectors represents how different their meanings are.
* **Combination**: We can combine multiple vectors to create new ones, which helps us understand complex relationships between words.

In short, vector embeddings help us represent words and phrases as single units that capture their meanings, allowing us to analyze and compare them in a more efficient way.


Role: chef | Topic: APIs


Imagine you're a chef, and you want to add a new recipe to your cookbook. You need to tell the world about it, so you write down the ingredients, instructions, and steps needed to make it.

An API is like a messenger who helps you share that information with others. When someone wants to use your recipe, they can send a request to the messenger (the API), saying "Hey, I want to know how to make this dish."

The messenger then checks if they have all the necessary ingredients and instructions written down. If they do, they send back the complete recipe to the person who requested it.

Just like how you might say "add 2 cups of flour" or "heat the oven to 350 degrees," APIs use special codes called "API calls" that convey what information is needed. When someone uses an API, they're essentially asking for specific data (like your recipe).

Here's a simple example:

**Your Recipe (Data)**

* Ingredients: 2 cups of flour, 1 cup of sugar, 1/2 teaspoon of salt
* Instructions: Preheat oven to 350 degrees. Mix ingredients together.

**API Call**

* Request: "Give me the ingredient list for this recipe"
* Response: "Here are the ingredients: flour (2 cups), sugar (1 cup), salt (1/2 teaspoon)"

In cooking, APIs help share recipes with others, but in real life, they're used to share information between different systems, like websites, apps, and services.

---

## 4. LCEL — LangChain Expression Language (The Pipe Operator)

LCEL is LangChain's **declarative composition syntax**. You chain components together using the pipe `|` operator — just like Unix pipes.

```
prompt | model | output_parser
```

Each component takes input from the previous one:

```
{variables} → PromptTemplate → Messages → ChatModel → AIMessage → StrOutputParser → string
```

The result is a **chain** — a new Runnable that supports `invoke()`, `stream()`, and `batch()`.

### Experiment 4A: Building Your First Chain with LCEL

In [9]:
# Three components snapped together with |
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "{question}")
])

parser = StrOutputParser()  # Extracts just the text from AIMessage

# Build the chain
chain = prompt | llm | parser

print(f"Chain type: {type(chain).__name__}")
print(f"Chain components: PromptTemplate | ChatOllama | StrOutputParser")

# invoke() — just pass the variables
result = chain.invoke({"question": "What are the 5 pillars of LangChain?"})

print(f"\nResult type: {type(result).__name__}")  # str, not AIMessage!
display(Markdown(result))

Chain type: RunnableSequence
Chain components: PromptTemplate | ChatOllama | StrOutputParser

Result type: TextAccessor


LangChain is an open-source framework that aims to simplify and accelerate the development of complex natural language processing (NLP) pipelines by providing a set of reusable, modular components.

The five pillars of LangChain are:

1. **Component Library**: This pillar focuses on building a comprehensive library of pre-trained, re-usable NLP models and components that can be easily integrated into other applications.
2. **Pipeline Framework**: This pillar provides a flexible framework for building complex NLP pipelines by allowing developers to define and execute tasks in a modular, chainable way.
3. **Task Management**: This pillar enables task automation and management through the use of a standardized interface for defining and executing tasks.
4. **Data Loading and Preprocessing**: This pillar provides tools and libraries for loading, preprocessing, and managing large datasets, making it easier to work with complex data sources.
5. **Integration and Extensibility**: This pillar focuses on providing a set of APIs and interfaces that allow developers to integrate LangChain components with other frameworks, libraries, and tools, while also enabling easy extension of the framework itself.

By providing these five pillars, LangChain aims to simplify the development of complex NLP pipelines by offering a flexible, modular, and scalable architecture for building and executing NLP tasks.

### Experiment 4B: Streaming Through a Chain

Because every component implements the Runnable protocol, the entire chain supports streaming automatically.

In [10]:
# Streaming works through the entire chain
print("Streaming through the chain:\n")

for chunk in chain.stream({"question": "What is LCEL in LangChain? Brief explanation."}):
    print(chunk, end="", flush=True)

print()

Streaming through the chain:

In LangChain, LCEL stands for "Language Chain Execution Layer". It's a layer that enables the execution of complex language models and NLP pipelines.

LCEL acts as an intermediate layer between the input data and the downstream model, allowing for more flexible and dynamic processing of natural language inputs. It provides a way to modularize and compose different components of a language pipeline, making it easier to build and deploy complex NLP applications.

Think of LCEL as a "orchestrator" that manages the flow of data through multiple layers of computation, allowing you to define complex workflows and transformations using a graph-based API.


### Experiment 4C: A More Useful Chain — Topic Explainer

In [11]:
# A reusable chain with multiple template variables
explainer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} instructor. Explain concepts clearly in {length}."),
    ("human", "Explain: {concept}")
])

explainer_chain = explainer_prompt | llm | StrOutputParser()

# Use it for different topics
topics = [
    {"domain": "AI/ML", "concept": "the Runnable protocol in LangChain", "length": "3 sentences"},
    {"domain": "software engineering", "concept": "why modular architecture matters", "length": "2 sentences"},
]

for topic in topics:
    print(f"\n{'=' * 50}")
    print(f"Concept: {topic['concept']}")
    print(f"{'=' * 50}")
    result = explainer_chain.invoke(topic)
    display(Markdown(result))


Concept: the Runnable protocol in LangChain


The `Runnable` protocol is a fundamental concept in LangChain, a Python library that provides a flexible framework for building custom workflows and pipelines.

In LangChain, a workflow is defined as a sequence of tasks that are executed one after another. Each task represents an operation that needs to be performed on some input data. The `Runnable` protocol defines the interface for a task that can be executed by LangChain's workflow engine.

A `Runnable` object in LangChain has two primary attributes:

1. **`run()`**: This is the method that gets called when the task is executed. It takes no arguments and returns no value (i.e., it has no output). The `run()` method performs the actual work of the task, such as reading data from a file, performing some computation, or making an API call.
2. **`inputs()`**: This attribute defines the input data required by the task. It is expected to be an iterable (e.g., list, tuple, generator) that contains the necessary inputs for the task to execute.

When a workflow is created in LangChain, each task is associated with a `Runnable` object. When the workflow engine executes a task, it calls the `run()` method of the corresponding `Runnable` object, passing in the input data defined by the `inputs()` attribute.

Here's an example of how to define a simple `Runnable` that reads a file and prints its contents:
```python
import langchain

class ReadFile(Runnable):
    def __init__(self, filename: str):
        self.filename = filename

    def run(self) -> None:
        with open(self.filename, 'r') as f:
            print(f.read())

# Create a workflow that executes the `ReadFile` task
workflow = langchain.Workflow(
    ReadFile('example.txt')
)

# Execute the workflow
workflow.run()
```
In this example, the `ReadFile` class implements the `Runnable` protocol by defining the `run()` method and the `inputs()` attribute. The `Run()` method reads the contents of a file and prints them to the console.

The `Runnable` protocol provides several benefits in LangChain, including:

* **Decoupling**: By separating tasks from their implementation details (i.e., the `Runnable` object), workflows can be designed without worrying about the underlying execution mechanism.
* **Reusability**: Tasks that implement the `Runnable` protocol can be reused across multiple workflows and contexts.
* **Flexibility**: LangChain's workflow engine can execute any task that implements the `Runnable` protocol, making it easy to integrate custom tasks into existing workflows.

Overall, the `Runnable` protocol is a fundamental building block of LangChain workflows, enabling developers to create flexible, reusable, and efficient pipelines for executing complex tasks.


Concept: why modular architecture matters


**Modular Architecture Matters for Sentence Transformers**

A modular architecture is essential for sentence transformer models, as it allows for several key benefits:

1. **Improved Scalability**: By breaking down the model into smaller, independent components (e.g., layers, blocks), sentence transformers can be scaled more efficiently. This means that increasing the model's capacity or complexity can be done incrementally, without requiring a complete retrain of the entire model.

2. **Faster Inference Times**: With modular architectures, each component can be computed independently, resulting in faster inference times for individual sentences. This is particularly important when working with large datasets and high-performance computing resources.

3. **Easier Maintenance and Updates**: When components are separate, it's easier to update or replace individual parts without affecting the entire model. For example, if a new pre-training task becomes available, the entire model might need to be retrained, but smaller components can still serve as frozen weights in other models, reducing overall training time.

4. **Reduced Overfitting**: Modular architectures often introduce regularization mechanisms (e.g., weight decay) that help prevent overfitting. By keeping each component small and independent, the risk of catastrophic forgetting or overfitting to specific tasks is reduced.

5. **Better Transferability**: With a modular architecture, it's easier to apply learned features from one task to another, leading to better transferability between different NLP applications.

In the context of sentence transformers specifically:

* The `multi-layer bidirectional transformer encoder` can be seen as a modular component, where each layer represents an independent transformation step.
* **Self-attention mechanisms** within the layers also operate independently, allowing for more efficient and flexible computations.

Some key advantages of modular architectures in sentence transformers include:

* Improved ability to leverage pre-trained models (e.g., BERT) as components
* Better handling of sequential data through efficient layer-by-layer processing

Overall, a well-designed modular architecture is crucial for achieving high performance, scalability, and maintainability in sentence transformer models.

---

## 5. Pillar 2 — Chains & Memory

LangChain provides **memory modules** to manage conversation history, since LLMs have no built-in memory.

| Memory Type | How It Works |
|-------------|--------------|
| **ConversationBufferMemory** | Stores the full conversation history |
| **ConversationBufferWindowMemory** | Keeps only the last K exchanges |
| **ConversationSummaryMemory** | LLM-generated summary of the conversation |

### Experiment 5A: Manual Conversation Memory

The simplest approach — manage the message list yourself.

In [12]:
# Manual conversation memory — you control the message list
conversation = [
    SystemMessage(content="You are a helpful AI tutor. Keep answers to 1-2 sentences.")
]

def chat_turn(user_input):
    """Add user message, get response, save to history."""
    conversation.append(HumanMessage(content=user_input))
    response = llm.invoke(conversation)
    conversation.append(response)  # AIMessage saved to history
    return response.content

# Multi-turn conversation
turns = [
    "What is LangChain?",
    "What problem does it solve?",
    "What was my first question?"
]

for i, user_msg in enumerate(turns, 1):
    print(f"\n{'=' * 50}")
    print(f"Turn {i}")
    print(f"{'=' * 50}")
    print(f"User: {user_msg}")
    answer = chat_turn(user_msg)
    print(f"AI:   {answer}")

print(f"\nTotal messages in history: {len(conversation)}")
print("The model remembers context because we pass the full history each time!")


Turn 1
User: What is LangChain?
AI:   LangChain is an open-source Python library that provides a simple, flexible, and efficient way to build and compose natural language processing (NLP) pipelines using pre-trained transformer models, such as BERT, RoBERTa, and others. It allows developers to easily integrate and chain together various NLP tasks, such as text classification, sentiment analysis, entity recognition, and question answering, into a single workflow.

Turn 2
User: What problem does it solve?
AI:   LangChain solves the problem of building complex natural language processing (NLP) pipelines by providing a unified interface for integrating multiple pre-trained transformer models, allowing developers to easily compose and chain together various NLP tasks without having to reinvent the wheel. This enables faster development, easier maintenance, and more efficient use of existing model capabilities.

Turn 3
User: What was my first question?
AI:   Your first question was "what is

### Experiment 5B: Conversation With a Window (Last K Turns)

In production, you can't grow the message list forever. A simple solution: keep only the last K exchanges.

In [14]:
def windowed_chat(messages_history, user_input, system_prompt, window_size=2):
    """Chat with a sliding window — keep only last K exchanges."""
    messages_history.append(HumanMessage(content=user_input))

    # Build the windowed context: system + last K pairs
    windowed = [SystemMessage(content=system_prompt)]
    # Keep last window_size * 2 messages (each exchange = human + ai)
    windowed.extend(messages_history[-(window_size * 2 + 1):])

    response = llm.invoke(windowed)
    messages_history.append(response)
    return response.content


# Test it — the model should "forget" early turns
history = []
system = "You are a helpful assistant. Keep answers to 1 sentence."

questions = [
    "My favorite color is blue. Remember that.",
    "What is 2 + 2?",
    "What is the capital of France?",
    "What is my favorite color?"  # Window=2 means this is likely forgotten
]

for i, q in enumerate(questions, 1):
    print(f"\nTurn {i}: {q}")
    print(f"History:{history}")
    answer = windowed_chat(history, q, system, window_size=2)
    print(f"AI:     {answer}")

print("\nWith window_size=2, the model only sees the last 2 exchanges.")
print("Earlier context (like favorite color) may be forgotten!")


Turn 1: My favorite color is blue. Remember that.
History:[]
AI:     I've taken note that your favorite color is blue.

Turn 2: What is 2 + 2?
History:[HumanMessage(content='My favorite color is blue. Remember that.', additional_kwargs={}, response_metadata={}), AIMessage(content="I've taken note that your favorite color is blue.", additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-03-01T05:02:31.3426764Z', 'done': True, 'done_reason': 'stop', 'total_duration': 538705800, 'load_duration': 93023800, 'prompt_eval_count': 475, 'prompt_eval_duration': 222562200, 'eval_count': 12, 'eval_duration': 177949000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019ca7c6-d192-7032-88f0-ea925cab7c53-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 475, 'output_tokens': 12, 'total_tokens': 487})]
AI:     The answer to 2 + 2 is 4.

Turn 3: What is the capital of France?
History:[HumanMessage(content='M

---

## 6. Pillar 3 — Swappability: The Power of Abstractions

LangChain's biggest value: **all providers implement the same interface**.

Switching from OpenAI to Ollama (or Claude, Gemini, etc.) requires changing **one line** — your chain logic stays the same.

```python
# Switch providers — chain logic is IDENTICAL
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_anthropic import ChatAnthropic

llm = ChatOpenAI(model="gpt-4o-mini")       # OpenAI
llm = ChatOllama(model="llama3.2:3b")        # Ollama (local)
llm = ChatAnthropic(model="claude-3-sonnet")  # Anthropic

# Same chain works with ANY of the above
chain = prompt | llm | parser
```

### Experiment 6A: Same Chain, Different Model Configs

In [16]:
# Demonstrate swappability with different Ollama configurations
# (Same concept applies when swapping between OpenAI, Anthropic, etc.)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Explain what a 'chain' is in LangChain. One sentence.")
])
parser = StrOutputParser()

# Config 1: Creative (high temperature)
creative_llm = ChatOllama(model=MODEL, temperature=1.0)
creative_chain = prompt | creative_llm | parser

# Config 2: Precise (low temperature)
precise_llm = ChatOllama(model=MODEL, temperature=0.0)
precise_chain = prompt | precise_llm | parser

print("SAME chain structure, DIFFERENT model configs:\n")

print("Creative (temp=1.0):")
print(creative_chain.invoke({}))

print("\nPrecise (temp=0.0):")
print(precise_chain.invoke({}))

print("\nThe chain logic (prompt | llm | parser) is identical.")
print("Only the model configuration changed!")

SAME chain structure, DIFFERENT model configs:

Creative (temp=1.0):
In LangChain, a "chain" refers to a hierarchical structure of language models and components that work together to achieve a specific NLP task or workflow, allowing users to create complex and scalable workflows by chaining multiple models and components together.

Precise (temp=0.0):
In LangChain, a "chain" refers to a sequence of tasks or operations that are executed one after another, allowing for modular and composable natural language processing workflows.

The chain logic (prompt | llm | parser) is identical.
Only the model configuration changed!


---

## 7. Output Parsers — Structured Output from LLMs

LLMs return raw text. **Output parsers** convert that text into structured data.

| Parser | Output |
|--------|---------|
| `StrOutputParser` | Plain string (strips the AIMessage wrapper) |
| `JsonOutputParser` | Parsed JSON dict |
| `CommaSeparatedListOutputParser` | Python list from comma-separated text |

### Experiment 7A: StrOutputParser vs Raw Response

In [17]:
# Without parser — you get an AIMessage object
raw_chain = prompt | llm
raw_result = raw_chain.invoke({})
print(f"Without parser: {type(raw_result).__name__}")
print(f"  Content: {raw_result.content}")

print()

# With parser — you get a clean string
parsed_chain = prompt | llm | StrOutputParser()
parsed_result = parsed_chain.invoke({})
print(f"With StrOutputParser: {type(parsed_result).__name__}")
print(f"  Content: {parsed_result}")

Without parser: AIMessage
  Content: In LangChain, a "chain" refers to a sequence of tasks or workflows that are executed in a pipeline-like fashion, allowing developers to compose complex NLP tasks from smaller, reusable components.

With StrOutputParser: TextAccessor
  Content: In LangChain, a "chain" refers to a sequence of tasks that are executed in a specific order, allowing for modular and composable workflows that can be easily combined and reused across different NLP applications.


### Experiment 7B: Comma-Separated List Parser

In [18]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()

list_prompt = ChatPromptTemplate.from_messages([
    ("system", "You list items separated by commas. No numbering, no extra text."),
    ("human", "List 5 {category}.")
])

list_chain = list_prompt | llm | list_parser

result = list_chain.invoke({"category": "programming languages"})

print(f"Type: {type(result).__name__}")  # list!
print(f"Result: {result}")
print(f"First item: {result[0]}")
print(f"Count: {len(result)}")

Type: list
Result: ['Python', 'JavaScript', 'Java', 'C++', 'Ruby']
First item: Python
Count: 5


### Experiment 7C: JSON Output Parser

In [19]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data assistant. Always respond with valid JSON only. "
     "No markdown, no explanation — just the JSON object."),
    ("human",
     "Give me info about {topic}. "
     'Return JSON with keys: "name", "category", "description" (1 sentence).')
])

json_chain = json_prompt | llm | json_parser

result = json_chain.invoke({"topic": "LangChain"})

print(f"Type: {type(result).__name__}")  # dict!
print(f"Result: {result}")

if isinstance(result, dict):
    for key, value in result.items():
        print(f"  {key}: {value}")

Type: dict
Result: {'name': 'LangChain', 'category': 'Open-source', 'description': 'A Python library for building and integrating various language models and AI pipelines.'}
  name: LangChain
  category: Open-source
  description: A Python library for building and integrating various language models and AI pipelines.


---

## 8. Pillar 4 & 5 — LangGraph & LangSmith (Overview)

### LangGraph — Agents with Graph-Based Control Flow

Chains are **linear**: Step A → Step B → Step C.

But real agents need **cycles**: reason → act → observe → decide to continue or try a different approach.

**LangGraph** provides:
- **State machines** — persistent state across steps
- **Nodes** — individual processing steps (LLM calls, tools, human input)
- **Edges** — connections with conditional routing
- **Human-in-the-loop** — pause for human approval

| Feature | LangChain Chains | LangGraph |
|---------|------------------|-----------|
| Flow | Linear | Cyclical, conditional |
| State | Stateless | Stateful + persistent |
| Control | Implicit | Explicit graph |
| Use case | Simple pipelines | Complex agents |

### LangSmith — Observability & Monitoring

When an LLM app gives a wrong answer — was it the prompt? The retrieved context? The model? LangSmith provides:

- **Tracing** — see every step of every chain execution
- **Evaluation** — automated testing of LLM outputs
- **Monitoring** — track latency, cost, errors in production
- **Datasets** — curate test cases for regression testing

> LangSmith integrates automatically — just set an environment variable and every execution is traced.

---

## 9. Putting It All Together — A Multi-Step Chain

Let's build a practical example that combines prompt templates, LCEL chains, and output parsing.

### Experiment 9A: Two-Step Chain — Generate then Analyze

In [20]:
# Step 1: Generate a concept explanation
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Explain concepts clearly in 2-3 sentences."),
    ("human", "Explain: {concept}")
])

explain_chain = explain_prompt | llm | StrOutputParser()

# Step 2: Simplify the explanation for a child
simplify_prompt = ChatPromptTemplate.from_messages([
    ("system", "You simplify technical explanations for a 10-year-old. Use an analogy. 2 sentences max."),
    ("human", "Simplify this: {explanation}")
])

simplify_chain = simplify_prompt | llm | StrOutputParser()

# Run them in sequence
concept = "vector embeddings"

print(f"Concept: {concept}")
print(f"\n{'=' * 50}")
print("Step 1 — Technical Explanation:")
print(f"{'=' * 50}")

explanation = explain_chain.invoke({"concept": concept})
display(Markdown(explanation))

print(f"\n{'=' * 50}")
print("Step 2 — Simplified for a 10-year-old:")
print(f"{'=' * 50}")

simple = simplify_chain.invoke({"explanation": explanation})
display(Markdown(simple))

Concept: vector embeddings

Step 1 — Technical Explanation:


**Vector Embeddings**

In natural language processing (NLP) and machine learning, a **vector embedding** is a mathematical representation of a word or token as a dense, fixed-length numerical vector. This vector captures the semantic meaning of the word in a high-dimensional space, allowing for more nuanced understanding of relationships between words.

**Key Characteristics:**

1. **Fixed Length**: Vector embeddings have a fixed length, typically ranging from 128 to 1024 dimensions.
2. **Dense Representation**: The embedding is a dense vector, meaning that each dimension corresponds to a specific feature or aspect of the word's meaning.
3. **Semantic Meaning**: The vector captures the semantic meaning of the word, such as its synonymy, antonymy, hyponymy, and collocation.

**How Vector Embeddings Work:**

1. **Word2Vec**: In Word2Vec, a popular algorithm for generating vector embeddings, words are represented as vectors in a high-dimensional space.
2. **Training**: The model is trained on large amounts of text data, where each word is mapped to its corresponding vector representation.
3. **Similarity Metrics**: Similarity between words is measured using distance metrics, such as cosine similarity or Euclidean distance.

**Types of Vector Embeddings:**

1. **Word2Vec**: A popular algorithm for generating vector embeddings, which uses a neural network to predict the context in which a word appears.
2. **GloVe**: A global vector representation (GloVe) algorithm that maps words to vectors based on their co-occurrence patterns in text data.
3. **FastText**: A variant of Word2Vec that allows for modeling out-of-vocabulary words and character-level representations.

**Advantages:**

1. **Improved Representation Capacity**: Vector embeddings can capture subtle nuances in word meaning, leading to better performance in NLP tasks.
2. **Efficient Computation**: Vector embeddings can be computed efficiently using matrix operations, making them suitable for large-scale applications.
3. **Robustness to Out-of-Vocabulary Words**: Many vector embedding algorithms can handle out-of-vocabulary words by utilizing techniques like character-level representations or masking.

**Common Applications:**

1. **Text Classification**: Vector embeddings are used in text classification tasks, such as spam detection and sentiment analysis.
2. **Named Entity Recognition**: Vector embeddings are used to represent named entities (e.g., people, organizations) for better understanding of their relationships.
3. **Question Answering**: Vector embeddings are used to represent questions and answers, allowing for more accurate matching and retrieval.

**Challenges:**

1. **Dimensionality Curse**: High-dimensional vector spaces can be computationally expensive to process.
2. **Overfitting**: Vector embeddings can suffer from overfitting if not regularized properly.
3. **Interpretability**: Understanding the meaning of vector embeddings can be challenging due to their abstract nature.

Overall, vector embeddings are a powerful tool for representing words in high-dimensional spaces, allowing for more nuanced understanding of relationships between words and improving performance in NLP tasks.


Step 2 — Simplified for a 10-year-old:


Here's an analogy to explain Vector Embeddings:

**Imagine Words as Colors**

Think of words like different colors on a palette. Just like how each color has its own unique properties (like brightness, saturation, and hue), words have their own meanings and associations.

**Vector Embeddings are Color Vectors**

When we create vector embeddings, we're essentially assigning numerical values to these word-colors. These numbers represent the "distance" or similarity between different colors (words). Just like how you can mix colors together to create new ones, our vector embeddings allow computers to understand the relationships between words.

**Key Features:**

* **Fixed Length**: Each color vector has a fixed length, which means we know exactly what's inside it.
* **Dense Representation**: The numbers in each vector tell us about specific aspects of the word's meaning (like its brightness or saturation).
* **Semantic Meaning**: These vectors capture the essence of each word, helping computers understand their relationships with other words.

**How Vector Embeddings Work:**

1. **Training**: We train our computer to learn these color-vector representations by showing it lots of text data.
2. **Similarity Metrics**: Our computer uses distance metrics (like distance between colors) to measure similarities between words.

**Types of Vector Embeddings:**

* **Word2Vec**: A popular algorithm that creates color vectors based on the context in which a word appears.
* **GloVe**: An algorithm that maps words to color vectors based on how often they appear together in text.
* **FastText**: A variant that allows for modeling out-of-vocabulary words and character-level representations.

**Advantages:**

1. **Improved Representation Capacity**: Vector embeddings help computers understand subtle nuances in word meaning, leading to better NLP performance.
2. **Efficient Computation**: Our computer can quickly compute vector embeddings using matrix operations.
3. **Robustness to Out-of-Vocabulary Words**: Many algorithms can handle unfamiliar words by utilizing techniques like character-level representations.

**Common Applications:**

1. **Text Classification**: Vector embeddings help computers classify text into categories (like spam vs. non-spam emails).
2. **Named Entity Recognition**: Our computer uses vector embeddings to represent named entities (like people or organizations) for better understanding of their relationships.
3. **Question Answering**: Vector embeddings aid in matching questions and answers, allowing for more accurate retrieval.

**Challenges:**

1. **Dimensionality Curse**: High-dimensional vector spaces can be computationally expensive to process.
2. **Overfitting**: Our computer might overfit if not regularized properly.
3. **Interpretability**: Understanding the meaning of vector embeddings can be challenging due to their abstract nature.

Overall, vector embeddings are a powerful tool for representing words in high-dimensional spaces, allowing computers to understand relationships between words and improve NLP performance.

### Experiment 9B: Chaining with RunnableLambda (Custom Logic in a Chain)

In [21]:
from langchain_core.runnables import RunnableLambda

# Custom function wrapped as a Runnable
def format_as_flashcard(text):
    """Format LLM output as a study flashcard."""
    lines = text.strip().split("\n")
    title = lines[0] if lines else "Concept"
    body = "\n".join(lines[1:]) if len(lines) > 1 else text
    return f"{'=' * 40}\n  FLASHCARD\n{'=' * 40}\n{text}\n{'=' * 40}"

# Build the chain: prompt → model → parse → custom format
flashcard_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Give a brief definition (1-2 sentences) then one key example."),
        ("human", "Define: {term}")
    ])
    | llm
    | StrOutputParser()
    | RunnableLambda(format_as_flashcard)  # Custom post-processing
)

terms = ["LCEL", "Prompt Template", "Output Parser"]

for term in terms:
    result = flashcard_chain.invoke({"term": term})
    print(result)
    print()

  FLASHCARD
**L CEL (Long Contextual Embeddings Learning)** is a deep learning approach used to learn contextualized embeddings for sentences or sequences of words.

The main idea behind L CEL is to use a multi-layer bidirectional neural network architecture, similar to those used in machine translation and natural language inference tasks. The network takes a sequence of tokens as input and outputs a fixed-length vector representation that captures the meaning of the entire sentence.

L CEL differs from traditional sentence embedding approaches like Sentence-BERT (sBERT) or Universal Sentence Encoder (USE), where the embedding layer is typically learned separately from the rest of the model. In contrast, L CEL uses an embedding layer as part of the neural network architecture itself, which allows for more efficient learning and better capture of contextual dependencies.

The key benefits of L CEL include:

* **Improved contextual understanding**: L CEL can learn to represent entire se

---

## 10. Package Installation Strategy

Install **only what you need**. The modular structure prevents dependency bloat.

| Use Case | Packages |
|----------|----------|
| Basic LLM apps (local) | `langchain langchain-core langchain-ollama` |
| Basic LLM apps (OpenAI) | `langchain langchain-core langchain-openai` |
| RAG applications | `langchain langchain-ollama langchain-chroma langchain-community` |
| Agent applications | `langgraph langchain-ollama` |
| Observability | `langsmith` |

```bash
# What we installed for this notebook:
pip install langchain langchain-core langchain-ollama
```

---

## 11. Sandbox — Try It Yourself!

In [ ]:
# ============================================================
#  SANDBOX - Edit and re-run!
# ============================================================

# Build your own chain
my_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep it brief."),
    ("human", "{question}")
])

my_chain = my_prompt | llm | StrOutputParser()

# Try different questions
question = "What makes LangChain different from calling an LLM API directly?"

use_streaming = True  # Toggle: True for streaming, False for invoke

# ============================================================

if use_streaming:
    print("[STREAMING]\n")
    for chunk in my_chain.stream({"question": question}):
        print(chunk, end="", flush=True)
    print()
else:
    print("[INVOKE]\n")
    result = my_chain.invoke({"question": question})
    display(Markdown(result))

---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **LangChain** | An ecosystem of 5 pillars — Core, LangChain, Community, LangGraph, LangSmith |
| **Core Abstractions** | Everything implements the Runnable protocol: `invoke()`, `stream()`, `batch()` |
| **Message Types** | `SystemMessage`, `HumanMessage`, `AIMessage` — structured, not raw dicts |
| **Prompt Templates** | Reusable, parameterized prompts with `ChatPromptTemplate` |
| **LCEL** | Compose chains with `\|` pipe operator: `prompt \| model \| parser` |
| **Output Parsers** | Convert raw LLM text to strings, lists, JSON, or custom formats |
| **Memory** | LLMs have no memory — you manage conversation history yourself |
| **Swappability** | Same chain works with any provider — change one import line |
| **Modular Install** | Install only what you need: `langchain-ollama`, `langchain-openai`, etc. |
| **LangGraph** | For stateful agents needing cycles, conditions, and persistence |
| **LangSmith** | Observability — tracing, evaluation, and monitoring for production |